# Time Series Cross-Validation in BoFire

This notebook demonstrates how to use time series cross-validation in BoFire to ensure that data from the same time series (e.g., batch experiments) stays together during cross-validation.

In [ ]:
import pandas as pd
import numpy as np
from bofire.data_models.domain.api import Inputs, Outputs
from bofire.data_models.features.api import (
    ContinuousInput,
    ContinuousOutput,
    CategoricalInput,
)
from bofire.data_models.surrogates.api import SingleTaskGPSurrogate
import bofire.surrogates.api as surrogates

## 1. Define Domain with Time Series Feature

The key is to set `is_timeseries=True` for the feature that identifies your time series groups (e.g., batch ID).

In [ ]:
# Define inputs with a batch identifier marked as time series
inputs = Inputs(
    features=[
        ContinuousInput(key="temperature", bounds=(20, 80)),
        ContinuousInput(key="pressure", bounds=(1, 10)),
        CategoricalInput(
            key="batch_id",
            categories=["A", "B", "C", "D"],
            is_timeseries=True,  # This marks it as the time series identifier!
        ),
    ]
)

outputs = Outputs(features=[ContinuousOutput(key="yield")])

print(f"Time series column: {inputs.get_timeseries_column()}")

## 2. Create Example Time Series Data

Each batch has multiple experiments. The key insight is that all experiments from the same batch should stay together during cross-validation.

In [ ]:
# Generate synthetic batch data
data = []
np.random.seed(42)

for batch in ["A", "B", "C", "D"]:
    # Each batch has 3 experiments at different conditions
    for _ in range(3):
        temp = np.random.uniform(20, 80)
        pressure = np.random.uniform(1, 10)
        # Simple yield model with batch effect
        batch_effect = {"A": 0, "B": 5, "C": 10, "D": 15}[batch]
        yield_val = 50 + batch_effect + 0.5 * (temp - 50) + 2 * (pressure - 5) + np.random.normal(0, 2)
        
        data.append({
            "temperature": temp,
            "pressure": pressure,
            "batch_id": batch,
            "yield": yield_val,
            "valid_yield": 1,
        })

experiments = pd.DataFrame(data)
experiments

## 3. Perform Cross-Validation with Automatic Time Series Detection

When calling `cross_validate`, the method will automatically detect `batch_id` as the time series column and use `GroupShuffleSplit` instead of random K-fold.

In [ ]:
# Create and map the surrogate model
model = SingleTaskGPSurrogate(inputs=inputs, outputs=outputs)
model = surrogates.map(model)

# Perform cross-validation - batch_id is automatically detected!
train_cv, test_cv, _ = model.cross_validate(experiments, folds=2)

print(f"Train R²: {train_cv.r2:.3f}")
print(f"Test R²: {test_cv.r2:.3f}")

## 4. Verify Batch Integrity

Let's verify that batches are kept together and not split between train and test sets.

In [ ]:
# Check which batches are in each fold
for fold_idx, (train_result, test_result) in enumerate(zip(train_cv.results, test_cv.results)):
    train_indices = list(train_result.observed.index)
    test_indices = list(test_result.observed.index)
    
    train_batches = experiments.iloc[train_indices]["batch_id"].unique()
    test_batches = experiments.iloc[test_indices]["batch_id"].unique()
    
    print(f"\nFold {fold_idx + 1}:")
    print(f"  Train batches: {sorted(train_batches)}")
    print(f"  Test batches: {sorted(test_batches)}")
    print(f"  ✓ No overlap: {len(set(train_batches) & set(test_batches)) == 0}")

## 5. Compare with Standard Random Cross-Validation

For comparison, let's see what happens if we manually override and don't use the time series grouping.

In [ ]:
# Create a model without time series flag
inputs_no_ts = Inputs(
    features=[
        ContinuousInput(key="temperature", bounds=(20, 80)),
        ContinuousInput(key="pressure", bounds=(1, 10)),
        CategoricalInput(
            key="batch_id",
            categories=["A", "B", "C", "D"],
            is_timeseries=False,  # Not marked as time series
        ),
    ]
)

model_no_ts = SingleTaskGPSurrogate(inputs=inputs_no_ts, outputs=outputs)
model_no_ts = surrogates.map(model_no_ts)

# This will use standard K-fold, potentially splitting batches
train_cv_random, test_cv_random, _ = model_no_ts.cross_validate(experiments, folds=2)

# Check if batches get split (they likely will)
for fold_idx, test_result in enumerate(test_cv_random.results):
    test_indices = list(test_result.observed.index)
    test_batches = experiments.iloc[test_indices]["batch_id"].value_counts()
    print(f"\nFold {fold_idx + 1} test set batch distribution:")
    print(test_batches)

## Summary

By setting `is_timeseries=True` on a feature:
- Cross-validation automatically uses `GroupShuffleSplit`
- Data from the same group (batch/time series) stays together
- This prevents data leakage and gives more realistic performance estimates

This is particularly important for:
- Batch experiments
- Time series data
- Any grouped/hierarchical experimental data